<a href="https://colab.research.google.com/github/alex-degarate/DAnalytics/blob/main/anexo/anexo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

================================================================================
# ANEXO                                

## PROBABLES MEJORAS A FUTURO        version 10 / 15 Dic 00:23


### *Trabajo en curso*



  

Se observó que:
1. Las ventas contienen productos con la misma denominación, sin ninguna id como para identificarlos. Pertenecen a distintas ventas individuales y tienen distinto precio.

2. Por tal razón es dificil hacer una evaluación seria del resultado de la campaña publicitaria, porque corresponde a distintos productos agrupados, con precio sin adjudicar.

3. La campaña de marketing posee el costo publicitario por producto (generico/agregado) pero no hay como relacionarlo con el costo_unitario, ni con el precio_venta

4. Sacar un precio_promedio, no estoy seguro que sirva mucho, porque la venta podria haberse hecho a un valor menor que el precio_unitario_promedio, debido a la gran dispersión de precios

Hemos visto que para el producto "Adorno de pared", los precios estan distribuidos en 3 segmentos o rangos, la idea es hacer un promedio por rango y
luego renombrar los productos segun el rango, por ej:
"Adorno de pared" => "Adorno de pared R1"

De esta manera en lugar de tener 100 productos genericos los tendriamos divididos por rango de precios y con un nuevo nombre_producto y un nuevo id_producto que los represente
  
.  




ESTE ES EL ENFOQUE QUE VOY A TRATAR DE DESARROLLAR

## df_producto
Contiene id_producto Unico para un "producto" clasificado por rangos: R1,
R2 y R3 porque los precios estan diferenciados  

Existe 2 denominaciones del producto:  
a) una generica, ej: "Adorno de pared"   
b) otra unica => "Adorno de pared-R1" con  su id asociado: 101  

Este DF contiene ademas:  
'rango': 'R1', 'R2', 'R3', respectivamente  
'media_rango': el valor promedio de ese producto en ese rango de precios  
'precio_rango': el rango de precio unitario para el producto en ese rango



## df_ventas
Los productos en df_ventas se vinculan por ID Unico, presente en ambas tablas  
Es opcional el nombre unico y estoy evaluando si lo pongo o no

Se agrego ["id_cat"] numerica para facilitar las operaciones y se mantiene["categoria"] para ser convertida como "category" de ser necesaria o borrada

Tambien se reordenaron los campos de los DF

## La salida de los graficos los tengo que borrar porque hay algo que hace que github invalide el notebook

In [ ]:
import pandas as pd

# habilito mostrar hasta 100 filas por DF
pd.set_option('display.max_rows', 100)

# Import dataset ventas final
url = "https://github.com/alex-degarate/DAnalytics/raw/refs/heads/main/anexo/"


In [ ]:
df_aLista_prod = pd.read_csv( url + "df_aLista_prod.csv", index_col=0)
df_aLista_prod

,Producto,Procesado,Count,id_prod
0,Adorno de pared,1,100,100
1,Alfombra,1,100,110
2,Aspiradora,1,100,120
3,Auriculares,1,143,130
4,Batidora,1,100,140
5,Cafetera,1,117,150
6,Candelabro,1,24,160
7,Consola de videojuegos,1,99,170
8,Cortinas,1,100,180
9,Cuadro decorativo,1,100,190


In [ ]:
# Import pre dataset ventas_market
df_ventas_market = pd.read_csv( url + "ventas_market.csv", index_col=0)

In [ ]:
# reindexa dejando el antiguo index
#df_ventas_market = df_ventas_market.reset_index()

# reindexa borrando el antiguo index
df_ventas_market = df_ventas_market.reset_index( drop=True)

In [ ]:
# Ensure date columns are in datetime format
df_ventas_market['fecha_venta'] = pd.to_datetime(df_ventas_market['fecha_venta'])  #, format="%d/%m/%Y")
df_ventas_market['fecha_venta'] = pd.to_datetime(df_ventas_market['fecha_venta'].dt.date)


In [ ]:
df_ventas_market['fecha_venta'].dtype

dtype('<M8[ns]')

In [ ]:
df_ventas_market.info()
df_ventas_market.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2998 entries, 0 to 2997
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_venta      2998 non-null   int64         
 1   id_prod       2998 non-null   int64         
 2   id_producto   2998 non-null   int64         
 3   producto      2998 non-null   object        
 4   rango         2998 non-null   object        
 5   precio_unit   2998 non-null   float64       
 6   cantidad      2998 non-null   int64         
 7   fecha_venta   2998 non-null   datetime64[ns]
 8   id_cat        2998 non-null   int64         
 9   categoria     2998 non-null   object        
 10  precio_rango  2998 non-null   object        
 11  dentro_camp   2998 non-null   int64         
 12  id_camp       2998 non-null   int64         
 13  canal         0 non-null      float64       
 14  costo_mrkt    0 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int6

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt
0,410,100,103,Adorno de pared,R3,109.64,3,2024-06-21,1,Decoración,91-120,0,0,NaN,NaN
1,620,100,103,Adorno de pared,R3,92.16,4,2024-10-21,1,Decoración,91-120,0,0,NaN,NaN
2,780,100,102,Adorno de pared,R2,79.13,7,2024-03-21,1,Decoración,66-91,0,0,NaN,NaN
3,50,100,102,Adorno de pared,R2,83.10,5,2024-01-31,1,Decoración,66-91,0,0,NaN,NaN
4,260,100,103,Adorno de pared,R3,101.48,9,2024-01-15,1,Decoración,91-120,0,0,NaN,NaN


In [ ]:
# Import dataset marketing
df_marketing = pd.read_csv( url + "df_marketing8.csv", index_col=0)


In [ ]:
df_marketing['fecha_ini'] = pd.to_datetime(df_marketing['fecha_ini'])
df_marketing['fecha_fin'] = pd.to_datetime(df_marketing['fecha_fin'])


In [ ]:
df_marketing['fecha_ini'] = pd.to_datetime(df_marketing['fecha_ini'].dt.date)
df_marketing['fecha_fin'] = pd.to_datetime(df_marketing['fecha_fin'].dt.date)


In [ ]:
#df_marketing['fecha_ini'] = pd.to_datetime(df_marketing['fecha_ini'], format="%d/%m/%Y")
#df_marketing['fecha_fin'] = pd.to_datetime(df_marketing['fecha_fin'], format="%d/%m/%Y")

In [ ]:
df_marketing.info()


<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 89
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id_camp     90 non-null     int64         
 1   producto    90 non-null     object        
 2   canal       90 non-null     object        
 3   costo_mrkt  90 non-null     float64       
 4   id_canal    90 non-null     int64         
 5   fecha_ini   90 non-null     datetime64[ns]
 6   fecha_fin   90 non-null     datetime64[ns]
 7   id_prod     90 non-null     int64         
 8   id_cat      90 non-null     int64         
 9   media_mrkt  90 non-null     float64       
dtypes: datetime64[ns](2), float64(2), int64(4), object(2)
memory usage: 7.7+ KB


In [ ]:
df_marketing.head()

,id_camp,producto,canal,costo_mrkt,id_canal,fecha_ini,fecha_fin,id_prod,id_cat,media_mrkt
0,14,Adorno de pared,RRSS,4.16,1,2024-10-22,2024-12-21,100,1,4.683333
1,74,Adorno de pared,TV,4.81,2,2024-03-20,2024-05-03,100,1,4.683333
2,44,Adorno de pared,Email,5.08,3,2024-04-13,2024-05-10,100,1,4.683333
3,58,Alfombra,Email,4.25,1,2024-03-31,2024-05-05,110,1,5.820000
4,28,Alfombra,RRSS,5.82,2,2024-11-27,2025-01-08,110,1,5.820000


In [ ]:
# Import dataset df_producto
df_producto = pd.read_csv( url + "df_producto7b.csv", index_col=0)

.  

=====

## Lista alfabetica de productos

In [ ]:
df_aLista_prod

,Producto,Procesado,Count,id_prod
0,Adorno de pared,1,100,100
1,Alfombra,1,100,110
2,Aspiradora,1,100,120
3,Auriculares,1,143,130
4,Batidora,1,100,140
5,Cafetera,1,117,150
6,Candelabro,1,24,160
7,Consola de videojuegos,1,99,170
8,Cortinas,1,100,180
9,Cuadro decorativo,1,100,190


In [ ]:
# Obtengo el codigo id_producto
#df_aLista_prod = pd.merge(df_aLista_prod, df_producto[['id_producto', 'producto_rango']], left_on='Producto', right_on='producto_rango', how='left')

# Drop the redundant 'producto_rango' column
#df_aLista_prod = df_aLista_prod.drop(columns=['producto_rango'])


In [ ]:
#df_aLista_prod.rename(columns={"id_producto": "id_prod"}, inplace=True)
#df_aLista_prod.head()

In [ ]:
# Guardamos df_aLista_prod
#df_aLista_prod.to_csv('df_aLista_prod.csv')

### tengo que agregar a ventas_market los datos que faltan
AGREGO EL CONTADOR PARA CADA PRODUCTO, a df_aLista_prod

================================================================================


# PASOS PARA AUTOMATIZAR
## PARA TODOS LOS PRODUCTOS

### 1. Recorrer La lista de productos y df_ventas_market,
&emsp; si el 2do campo esta en cero significa que no fue procesado, 1=completo
lista_productos = [["Adorno de pared", 1]

### 2. OBTENER NOMBRE DEL PRODUCTO A PROCESAR


### 9. ACTUALIZAR df_ventas_market


## 2. OBTENER NOMBRE DEL PRODUCTO A PROCESAR

In [ ]:
# @title

nLenght = df_aLista_prod.shape[0]
'''
df_aLista_prod.iat[ 0, 1] = 1  # Adorno de pared
df_aLista_prod.iat[ 1, 1] = 1  # Alfombra
df_aLista_prod.iat[ 2, 1] = 1  # Aspiradora
df_aLista_prod.iat[ 3, 1] = 1  # Auriculares
df_aLista_prod.iat[ 4, 1] = 1  # Batidora
df_aLista_prod.iat[ 5, 1] = 1  # Cafetera
df_aLista_prod.iat[ 6, 1] = 1  # Candelabro
df_aLista_prod.iat[ 7, 1] = 1  # Consola de videojuegos
df_aLista_prod.iat[ 8, 1] = 1  # Cortinas
df_aLista_prod.iat[ 9, 1] = 1  # Cuadro decorativo
df_aLista_prod.iat[10, 1] = 1  # Cámara digital
df_aLista_prod.iat[11, 1] = 1  # Elementos de cerámica
df_aLista_prod.iat[12, 1] = 1  # Espejo decorativo
df_aLista_prod.iat[13, 1] = 1  # Freidora eléctrica
df_aLista_prod.iat[14, 1] = 1  # Heladera
df_aLista_prod.iat[15, 1] = 1  # Horno eléctrico
df_aLista_prod.iat[16, 1] = 1  # Jarrón decorativo
df_aLista_prod.iat[17, 1] = 1  # Laptop
df_aLista_prod.iat[18, 1] = 1  # Lavadora
df_aLista_prod.iat[19, 1] = 1  # Lámpara de mesa
df_aLista_prod.iat[20, 1] = 1  # Microondas
df_aLista_prod.iat[21, 1] = 1  # Parlantes Bluetooth
df_aLista_prod.iat[22, 1] = 1  # Plancha de vapor
df_aLista_prod.iat[23, 1] = 1  # Proyector
df_aLista_prod.iat[24, 1] = 1  # Rincón de plantas
df_aLista_prod.iat[25, 1] = 1  # Secadora
df_aLista_prod.iat[26, 1] = 1  # SmartWatch
df_aLista_prod.iat[27, 1] = 1  # Smartphone
df_aLista_prod.iat[28, 1] = 1  # Tablet
#df_aLista_prod.iat[29, 1] = 1  # Televisor

for i in range( nLenght):

    if df_aLista_prod.iat[i, 1] == 0:
       prod_actual = df_aLista_prod.iat[i, 0]      # nombre producto
       break

prod_actual = prod_actual
'''

num_elem = df_aLista_prod.iat[i, 2]
print(f"El primer item a procesar es {[i]}: {prod_actual}, tiene {num_elem} registros \n")


El primer item a procesar es [29]: Televisor, tiene 100 registros 



In [ ]:
df_aLista_prod.iat[19, 1] = 0 # Lámpara de mesa
display(df_aLista_prod)

# df_ventas_market ACTUAL

In [ ]:
# agrego un campo nuevo
df_ventas_market['id_canal'] = 0

In [ ]:
# pongo campo [canal]="no"y	[costo_mrkt] =0.00
len_ventas_mrkt = df_ventas_market.shape[0]
print(f"len_ventas_mrkt = {len_ventas_mrkt}")

for i in range( len_ventas_mrkt):
    df_ventas_market.iat[i, 11] = 0      # dentro_camp
    df_ventas_market.iat[i, 12] = 0      # id_camp
    df_ventas_market.iat[i, 13] = "no"   # canal
    df_ventas_market.iat[i, 14] = 0.00   # costo_mrkt
    df_ventas_market.iat[i, 15] = 0.00   # id_canal

len_ventas_mrkt = 2998


In [ ]:
#df_ventas_market.info()
df_ventas_market.head(50)

In [ ]:
df_ventas_market['fecha_venta'].tail(5)

In [ ]:
import numpy as np

# Agregar a df_ventas_market los datos faltantes
cont_prod = 0  # i

rec_vent_mrkt = 0

#fecha_venta = df_ventas_market.iat[j, 7]  # fecha_venta posic 7
#print( f"fecha_venta {type(df_ventas_market.iat[j, 7])}")
#print( f"{type(df_ventas_market.iat[j, 7])}")

for i in range( 1):  #nLenght es el numero de productos

    prod_actual = df_aLista_prod.iat[i, 0]      # nombre producto

    # obtengo los 3 registros canal df_marketing de canales para prod_actual
    df_canal_mrkt = df_marketing[df_marketing['producto'] == prod_actual]

    display(df_canal_mrkt)
    print("\n\n")
    # si fecha_venta esta contenida entre fecha_ini y fecha_fin
    # obtengo campo id_camp, canal,  costo_mrkt

    # j= rec_vent_mrkt


    # Para todo df_ventas_mrkt ----------------------
    for j in range( len_ventas_mrkt): # len_ventas_mrkt

        #obtengo fecha venta
        fecha_venta = df_ventas_market.iat[j, 7]
        #print( f"{type(df_ventas_market.iat[j, 7])}")


        rec_mrkt = 0
        # para cada canal, chequeo el registro de ventas
        while rec_mrkt < 3:

              if (df_canal_mrkt.iat[rec_mrkt, 5]  <= fecha_venta) & (fecha_venta <=
                 df_canal_mrkt.iat[rec_mrkt, 6]):

                  # faltaria controlar las campañas que se solapen
                  df_canal_mrkt.iat[rec_mrkt, 0] # dentro_camp
                  df_ventas_market.iat[j,11] = 1
                  df_ventas_market.iat[j,12] = df_canal_mrkt.iat[rec_mrkt, 0] # id_camp
                  df_ventas_market.iat[j,13] = df_canal_mrkt.iat[rec_mrkt, 2] # canal
                  df_ventas_market.iat[j,14] = df_canal_mrkt.iat[rec_mrkt, 3] # costo_mrkt
                  df_ventas_market.iat[j,15] = df_canal_mrkt.iat[rec_mrkt, 4] # id_canal
                  print(f"  {rec_mrkt} encontrado ! id_camp: {df_canal_mrkt.iat[rec_mrkt, 0]}, canal: {df_canal_mrkt.iat[rec_mrkt, 2]}, costo_mrkt: {df_canal_mrkt.iat[rec_mrkt, 3]}")

              else:
                  # los valores por defecto ya estan puestos...
                  print(f"  {rec_mrkt} Not found ! => {fecha_venta}")

              rec_mrkt += 1

print("\nLlego al final!")

,id_camp,producto,canal,costo_mrkt,id_canal,fecha_ini,fecha_fin,id_prod,id_cat,media_mrkt
0,14,Adorno de pared,RRSS,4.16,1,2024-10-22,2024-12-21,100,1,4.683333
1,74,Adorno de pared,TV,4.81,2,2024-03-20,2024-05-03,100,1,4.683333
2,44,Adorno de pared,Email,5.08,3,2024-04-13,2024-05-10,100,1,4.683333


Se truncaron las últimas líneas 5000 del resultado de transmisión.
  0 Not found ! => 2024-06-26 00:00:00
  1 Not found ! => 2024-06-26 00:00:00
  2 Not found ! => 2024-06-26 00:00:00
  0 Not found ! => 2024-04-29 00:00:00
  1 encontrado ! id_camp: 74, canal: TV, costo_mrkt: 4.81
  2 encontrado ! id_camp: 44, canal: Email, costo_mrkt: 5.08
  0 Not found ! => 2024-05-20 00:00:00
  1 Not found ! => 2024-05-20 00:00:00
  2 Not found ! => 2024-05-20 00:00:00
  0 Not found ! => 2024-05-26 00:00:00
  1 Not found ! => 2024-05-26 00:00:00
  2 Not found ! => 2024-05-26 00:00:00
  0 Not found ! => 2024-04-30 00:00:00
  1 encontrado ! id_camp: 74, canal: TV, costo_mrkt: 4.81
  2 encontrado ! id_camp: 44, canal: Email, costo_mrkt: 5.08
  0 Not found ! => 2024-09-24 00:00:00
  1 Not found ! => 2024-09-24 00:00:00
  2 Not found ! => 2024-09-24 00:00:00
  0 encontrado ! id_camp: 14, canal: RRSS, costo_mrkt: 4.16
  1 Not found ! => 2024-10-27 00:00:00
  2 Not found ! => 2024-10-27 00:00:00
  0 Not fou

In [ ]:
# Guardamos df_ventas_market
#df_ventas_market.to_csv('df_ventas_market.csv')

In [ ]:
# Guardamos df_marketing
# df_marketing.to_csv('df_marketing.csv')

GUARDAR UNA COPIA EN ARCHIVO TEXTO

In [ ]:
# @title
import io
import sys

# Create a StringIO object to capture output
capture_output = io.StringIO()

# Redirect standard output to the StringIO object
sys.stdout = capture_output

# --- Start of your loop/logic where print statements occur ---
# For demonstration, I'll use a simplified loop structure similar to yours.
# You would place your actual processing logic (the loops from EN_9XpzdFwlm)
# here, and its print statements will be captured.

# Example values for demonstration
demo_prod_actual = "Adorno de pared"
demo_fecha_venta = pd.Timestamp('2024-04-15 00:00:00')
demo_df_canal_mrkt = pd.DataFrame({
    'id_camp': [14, 74, 44],
    'producto': ['Adorno de pared', 'Adorno de pared', 'Adorno de pared'],
    'canal': ['RRSS', 'TV', 'Email'],
    'costo_mrkt': [4.16, 4.81, 5.08],
    'id_canal': [1, 2, 3],
    'fecha_ini': [pd.Timestamp('2024-10-22'), pd.Timestamp('2024-03-20'), pd.Timestamp('2024-04-13')],
    'fecha_fin': [pd.Timestamp('2024-12-21'), pd.Timestamp('2024-05-03'), pd.Timestamp('2024-05-10')]
})

output_messages = []

output_messages.append(f"Campaigns for {demo_prod_actual}:\n{demo_df_canal_mrkt.to_string()}\n\n")

for rec_mrkt in range(len(demo_df_canal_mrkt)):
    campaign_ini = demo_df_canal_mrkt.iat[rec_mrkt, demo_df_canal_mrkt.columns.get_loc('fecha_ini')]
    campaign_fin = demo_df_canal_mrkt.iat[rec_mrkt, demo_df_canal_mrkt.columns.get_loc('fecha_fin')]

    if (campaign_ini <= demo_fecha_venta) and (demo_fecha_venta <= campaign_fin):
        output_messages.append(f"  {rec_mrkt} encontrado ! id_camp: {demo_df_canal_mrkt.iat[rec_mrkt, demo_df_canal_mrkt.columns.get_loc('id_camp')]}, canal: {demo_df_canal_mrkt.iat[rec_mrkt, demo_df_canal_mrkt.columns.get_loc('canal')]}, costo_mrkt: {demo_df_canal_mrkt.iat[rec_mrkt, demo_df_canal_mrkt.columns.get_loc('costo_mrkt')]}")
    else:
        output_messages.append(f"  {rec_mrkt} Not found ! => {demo_fecha_venta}")

output_messages.append("\nSimplified demo finished!")

# Print the messages using the captured stdout
for msg in output_messages:
    print(msg)

# --- End of your loop/logic ---

# Restore standard output to the console
sys.stdout = sys.__stdout__

# Get the captured output as a string
captured_string = capture_output.getvalue()

print("\n--- Captured Output (stored in 'captured_string' variable) ---")
print(captured_string)

# You can also write this to a file:
# with open('captured_log.txt', 'w') as f:
#     f.write(captured_string)


In [ ]:
# @title
'''
import numpy as np

# Assuming prod_actual and df_canal_mrkt (filtered df_marketing for prod_actual) are already defined.
# For example, if prod_actual = 'Televisor':
# df_canal_mrkt = df_marketing[df_marketing['producto'] == prod_actual].copy()

def get_campaign_details(sale_row, df_product_campaigns):
    sale_date = sale_row['fecha_venta']

    # Filter campaigns to find those active during the sale date
    active_campaigns = df_product_campaigns[
        (df_product_campaigns['fecha_ini'] <= sale_date) &
        (df_product_campaigns['fecha_fin'] >= sale_date)
    ]

    if not active_campaigns.empty:
        # If multiple campaigns are active, you might need specific logic to pick one.
        # For now, we take the first one found.
        campaign = active_campaigns.iloc[0]
        return pd.Series({
            'dentro_camp': 1,
            'id_camp': campaign['id_camp'],
            'canal': campaign['canal'],
            'costo_mrkt': campaign['costo_mrkt']
        })
    else:
        return pd.Series({
            'dentro_camp': 0,
            'id_camp': 0, # Use 0 or np.nan to indicate no campaign
            'canal': np.nan,
            'costo_mrkt': np.nan
        })

# Filter df_ventas_market for the current product to avoid updating all rows unnecessarily
# We use .loc and .copy() to ensure we are working on a copy and not a view
sales_for_prod_actual = df_ventas_market.loc[df_ventas_market['producto'] == prod_actual].copy()

# Apply the function to each row of the filtered sales data
campaign_info = sales_for_prod_actual.apply(
    lambda row: get_campaign_details(row, df_canal_mrkt),
    axis=1
)

# Update the original df_ventas_market with the new campaign information
df_ventas_market.loc[sales_for_prod_actual.index, ['dentro_camp', 'id_camp', 'canal', 'costo_mrkt']] = campaign_info

print(f"Updated df_ventas_market for product: {prod_actual}")
display(df_ventas_market[df_ventas_market['producto'] == prod_actual].head())
'''

EMPEZAR CON METRICAS

In [ ]:
# Transformación:
# Calculamos el valor de cada venta = precio_unit * cantidad

# Agregamos una nueva columna  valor_venta = monto
df_ventas_market["valor_venta"] = df_ventas_market["precio_unit"] * df_ventas_market["cantidad"]


In [ ]:
# Agregamos una nueva columna  gasto_venta (si no hay campaña. el costo es cero )
df_ventas_market["gasto_venta"] = df_ventas_market["costo_mrkt"] * df_ventas_market["cantidad"]

In [ ]:
# Generamos primero la columna mes
df_ventas_market["mes"] = df_ventas_market["fecha_venta"].dt.month

In [ ]:
df_ventas_market.head()

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,mes
0,410,100,103,Adorno de pared,R3,109.64,3,2024-06-21,1,Decoración,91-120,0,0,no,0.00,0,328.92,0.00,6
1,620,100,103,Adorno de pared,R3,92.16,4,2024-10-21,1,Decoración,91-120,0,0,no,0.00,0,368.64,0.00,10
2,780,100,102,Adorno de pared,R2,79.13,7,2024-03-21,1,Decoración,66-91,1,74,TV,4.81,2,553.91,33.67,3
3,50,100,102,Adorno de pared,R2,83.10,5,2024-01-31,1,Decoración,66-91,0,0,no,0.00,0,415.50,0.00,1
4,260,100,103,Adorno de pared,R3,101.48,9,2024-01-15,1,Decoración,91-120,0,0,no,0.00,0,913.32,0.00,1


In [ ]:
df_ventas_market.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2998 entries, 0 to 2997
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_venta      2998 non-null   int64         
 1   id_prod       2998 non-null   int64         
 2   id_producto   2998 non-null   int64         
 3   producto      2998 non-null   object        
 4   rango         2998 non-null   object        
 5   precio_unit   2998 non-null   float64       
 6   cantidad      2998 non-null   int64         
 7   fecha_venta   2998 non-null   datetime64[ns]
 8   id_cat        2998 non-null   int64         
 9   categoria     2998 non-null   object        
 10  precio_rango  2998 non-null   object        
 11  dentro_camp   2998 non-null   int64         
 12  id_camp       2998 non-null   int64         
 13  canal         2998 non-null   object        
 14  costo_mrkt    2998 non-null   float64       
 15  id_canal      2998 non-null   int64   

In [ ]:
# Iniciamos un DF con datos de cada producto
df_estad_prod = df_aLista_prod.copy()

Hacer monto total de todos los productos

Hacer un df para informe final con datos del producto,
- ventas totales de ese producto
- unidades vendidas en cada canal

In [ ]:
df_estad_prod.head()

,Producto,Procesado,Count,id_prod
0,Adorno de pared,1,100,100
1,Alfombra,1,100,110
2,Aspiradora,1,100,120
3,Auriculares,1,143,130
4,Batidora,1,100,140


## Total de ventas

In [ ]:
ventas_totales = df_ventas_market["valor_venta"].sum()
print(f"El total de ventas es: ${ventas_totales: ,.02f}")

El total de ventas es: $ 1,467,093.52


In [ ]:
num_con_camp = df_ventas_market[df_ventas_market['dentro_camp'] == 1].shape[0]
num_sin_camp = df_ventas_market[df_ventas_market['dentro_camp'] == 0].shape[0]
print(f"numero Total de operaciones comerciales:")
print(f"num_con_camp = {num_con_camp}")
print(f"num_sin_camp = {num_sin_camp}")

#total_ventas_cant = df_ventas_market["cantidad"].sum()
#print(f"total_ventas_cant = {total_ventas_cant}")

# total por cantidad de registros, no por cantidad de unidades
num_total = num_con_camp + num_sin_camp
print(f"numero Total = {num_total}")

numero Total de operaciones comerciales:
num_con_camp = 954
num_sin_camp = 2044
numero Total = 2998


Cantidad Operaciones comerciales

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data for the pie chart
label1 = 'Con Campaña' + ' (' + str(num_con_camp) + ')'
label2 = 'Sin Campaña' + ' (' + str(num_sin_camp) + ')'
labels = [label1, label2]
sizes = [num_con_camp, num_sin_camp]
#colors = ['#66b3ff', '#ff9999']
colors = ['#ff7f0e', '#2ca02c']
explode = (0.1, 0)

plt.figure(figsize=(4, 4))
plt.pie(sizes, explode=explode, labels=labels, colors=colors,
        autopct='%1.1f%%', shadow=True, startangle=140)

plt.axis('equal')
plt.title('Operaciones Comerciales Con y Sin Campaña [registros]', fontsize=14)

# Añadir una anotación para el total de unid vendidas en la parte inferior
plt.text(1.7, 0.10, f'Total Ventas Cantidad: ${num_total:.0f}', transform=plt.gca().transAxes,
         fontsize=11, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

plt.show()

In [ ]:
# Agrupamos por categoria y agregamos el total valor_venta por categoria
# (solo son 3 categorías )
ventas_categoria = df_ventas_market.groupby("categoria", as_index=False)["valor_venta"].sum()

# Ordenar y mostrar los resultados
ventas_categoria.sort_values(by="valor_venta", ascending=False, inplace=True)
ventas_categoria.head()

,categoria,valor_venta
1,Electrodomésticos,505299.63
2,Electrónica,482577.80
0,Decoración,479216.09


Dividir ventas por los que estan con marketing y sin marketing

In [ ]:
# Filtrar df_ventas_market para ver solamente  ventas_con_marketing
ventas_con_market = df_ventas_market[df_ventas_market['dentro_camp'] == 1]

In [ ]:
#print("ventas_con_marketing:")
display(ventas_con_market.head())

## BORRAR ?

In [ ]:
# @title
'''
# Entonces agrupamos primero por id_venta y producto
# de esa particion, nos quedamos con la primera venta y la suma de los costos de marketing
ventas_con_marketing_agg1 = ventas_con_market.groupby(["producto"], as_index=False).agg(
    valor_venta=("valor_venta", "suma"),
    costo_agg=("costo_mrkt", "sum")
)

# Verificamos la consistencia de los montos, ventas y costos de marketing
ventas_con_marketing_agg1[["id_venta","producto", "valor_venta", "gasto_venta"]].sort_values(by=["id_venta"], ascending=True).head(6)
'''

In [ ]:
# Agregar una nueva columna para diferencia: valor_venta - gasto_venta
df_ventas_market['margen_neto'] = df_ventas_market['valor_venta'] - df_ventas_market['gasto_venta']


In [ ]:
# Group by 'producto' and sum the 'margen_neto_marketing' column
#productos_margen_neto
ganancia_con_market = df_ventas_market.groupby('producto', as_index=False)['margen_neto'].sum()

# ordenados por producto
display(ganancia_con_market )

In [ ]:
ganancia_con_market.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   producto     30 non-null     object 
 1   margen_neto  30 non-null     float64
dtypes: float64(1), object(1)
memory usage: 612.0+ bytes


In [ ]:
# mostrar resultados ordenados por margen de ganancia
print("Margen neto de marketing por producto:")
display(ganancia_con_market.sort_values(by='margen_neto', ascending=False).head(7))


In [ ]:
# @title
ganancia_cm = ganancia_con_market.sort_values(by='margen_neto', ascending=False).head(5)
print(len(ganancia_cm))

# PRODUCTOS TOP POR MONTO
import seaborn as sns
import matplotlib.pyplot as plt

# Crear el gráfico de torta
plt.figure(figsize=(4, 4))
sns.set_style("whitegrid")

explode = [0.1, 0,0,0,0]

# The original 'explode' variable from a previous cell had 8 elements.
# 'prod_top_cant' has 7 products (check prod_top_cant.shape[0] or len(prod_top_cant)).
# So, 'explode' must have a length of 7 to match.
#explode = [0.1] + [0] * (len(ganancia_con_market) - 1)
#explode = [0.1] + [0] * (4)

# ['prod_gener'] = "producto" en la otra tabla
wedges, texts, autotexts = plt.pie(ganancia_cm['margen_neto'], labels=ganancia_cm['producto'],
                                   autopct='%1.1f%%', explode=explode, startangle=90)

# Cambiar los atributos del título
plt.title('Venta Productos Top por Monto', fontsize=14, color='darkblue', style='italic')

plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

# Cambiar el color de los porcentajes a blanco
for autotext in autotexts:
    autotext.set_color('white')

plt.show()


In [ ]:
# Filtrar df_ventas_market pars ver solamente  ventas_sin_marketing
ventas_sin_market = df_ventas_market[df_ventas_market['dentro_camp'] == 0]
ventas_sin_market.tail()

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,margen_neto,mes
2990,2092,390,392,Televisor,R2,65.23,6,2024-05-25,3,Electrónica,57-85,0,0,no,0.0,0,391.38,0.0,391.38,5
2993,2652,390,392,Televisor,R2,74.49,4,2024-05-25,3,Electrónica,57-85,0,0,no,0.0,0,297.96,0.0,297.96,5
2994,2252,390,393,Televisor,R3,89.27,8,2024-01-26,3,Electrónica,85-124,0,0,no,0.0,0,714.16,0.0,714.16,1
2995,2082,390,391,Televisor,R1,55.66,3,2024-03-15,3,Electrónica,26-57,0,0,no,0.0,0,166.98,0.0,166.98,3
2996,2991,390,393,Televisor,R3,92.33,4,2024-07-15,3,Electrónica,85-124,0,0,no,0.0,0,369.32,0.0,369.32,7


In [ ]:
# Group by 'producto' and sum the 'margen_neto_marketing' column
#productos_margen_neto
ganancia_sin_market = ventas_sin_market.groupby('producto', as_index=False)['margen_neto'].sum()

# ordenados por producto
display(ganancia_sin_market.head() )

,producto,margen_neto
0,Adorno de pared,33055.59
1,Alfombra,27735.03
2,Aspiradora,34065.35
3,Auriculares,51612.29
4,Batidora,31710.67


In [ ]:
ventas_rrss = ventas_con_market[ventas_con_market['canal'] == 'RRSS']
ventas_rrss.head(3)

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,margen_neto,mes
8,640,100,103,Adorno de pared,R3,110.44,7,2024-12-02,1,Decoración,91-120,1,14,RRSS,4.16,1,773.08,29.12,743.96,12
17,130,100,102,Adorno de pared,R2,68.70,3,2024-12-08,1,Decoración,66-91,1,14,RRSS,4.16,1,206.10,12.48,193.62,12
31,690,100,103,Adorno de pared,R3,103.95,8,2024-10-26,1,Decoración,91-120,1,14,RRSS,4.16,1,831.60,33.28,798.32,10


In [ ]:
ventas_tv = ventas_con_market[ventas_con_market['canal'] == 'TV']
ventas_tv.head(3)

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,margen_neto,mes
2,780,100,102,Adorno de pared,R2,79.13,7,2024-03-21,1,Decoración,66-91,1,74,TV,4.81,2,553.91,33.67,520.24,3
7,950,100,103,Adorno de pared,R3,116.42,6,2024-03-20,1,Decoración,91-120,1,74,TV,4.81,2,698.52,28.86,669.66,3
10,600,100,101,Adorno de pared,R2,66.00,4,2024-04-10,1,Decoración,25-66,1,74,TV,4.81,2,264.00,19.24,244.76,4


In [ ]:
ventas_email = ventas_con_market[ventas_con_market['canal'] == 'Email']
ventas_email.head(3)

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,margen_neto,mes
9,1000,100,102,Adorno de pared,R2,74.64,11,2024-05-08,1,Decoración,66-91,1,44,Email,5.08,3,821.04,55.88,765.16,5
16,550,100,103,Adorno de pared,R3,98.99,8,2024-05-06,1,Decoración,91-120,1,44,Email,5.08,3,791.92,40.64,751.28,5
21,650,100,101,Adorno de pared,R1,43.11,4,2024-05-05,1,Decoración,25-66,1,44,Email,5.08,3,172.44,20.32,152.12,5


In [ ]:
# @title
# # **ARREGLAR !**!@title
ventas_con_sin_marketing = pd.merge(ganancia_con_market, ganancia_sin_market, on="producto", how="inner")

#ventas_con_sin_marketing = ventas_con_sin_marketing.sort_values(by=["ventas_producto_cm"], ascending=False)

ventas_con_sin_marketing.head()


In [ ]:
# @title
# Transformar a formato largo
df_long = ventas_con_sin_marketing[["producto","ganancia_con_market", "ganancia_sin_market"]]
df_long = df_long.melt(id_vars="producto",
                        var_name="modalidad",
                        value_name="valores")

# df_long = df_long.sort_values(by=["valores", "modalidad"], ascending=True)
df_long.head()

## Graficar comparativa con y sin marketing

In [ ]:
# @title
# Estilo y paleta
sns.set_theme(               # set_theme combina estilo + contexto + paleta
    style="whitegrid",       # opciones: 'white', 'whitegrid', 'dark', 'darkgrid', 'ticks'
    context="notebook",          # escala general: 'paper', 'notebook', 'talk', 'poster'
    palette="tab10"           # paleta de colores base: "deep", "muted", "pastel" / Set1, Set2, Set3
)

plt.figure(figsize=(9,5))

colores = sns.color_palette('tab10').as_hex()
colores[0] = '#2ca02c'
colores[1] = '#ffb366'
#sns.palplot(sns.color_palette(colores))

sns.set_theme( style="whitegrid", palette=sns.color_palette(colores))

sns.barplot(data=df_long, x="producto", y="valores", hue="modalidad", width=0.8)

# sns.barplot(data=df_long, x="Trimestre", y="Ventas", hue="Producto", width=0.8)
#Invertimos Productos por Trimestre

# Ticks (valores de eje)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)

# Ajustes de título y ejes
plt.title('\nVentas totales por productos dentro y fuera de campaña', fontsize=14)
plt.xlabel('Productos')
plt.ylabel('Ventas')

# Obtener handles (cuadritos de color) y labels originales del gráfico
handles, labels = plt.gca().get_legend_handles_labels()

# Reemplazar solo los textos de la leyenda, manteniendo colores
plt.legend(
    handles,
    ["Fuera de campaña", "Dentro de campaña"],
    title="Modalidad",
    fontsize=9,
    title_fontsize=10
)
plt.tight_layout()
plt.show()


##

# ====================================================

## Guardamos los dataframe en uso...

In [ ]:
# Guardamos df_ventas_market
#df_ventas_market.to_csv('df_ventas_market.csv')

In [ ]:
# Guardamos df_marketing
#df_marketing.to_csv('df_marketing.csv')


In [ ]:
# Guardamos df_aLista_prod
#df_producto.to_csv('df_aLista_prod.csv', index=True)

.  

===============================================================================

## A PARTIR DE ACA EL REPROCESAMIENTO DE VENTAS Y MARKETING 2  


Para maximizar el tiempo estoy arrancando desde el dataframe armado previamente y guardado en disco.

In [ ]:
# Import pre dataset ventas_market ya procesado
df_ventas_market2 = pd.read_csv( url + "df_ventas_market10.csv", index_col=0)


In [ ]:
# Ensure date columns are in datetime format
df_ventas_market2['fecha_venta'] = pd.to_datetime(df_ventas_market2['fecha_venta'])  #, format="%d/%m/%Y")
df_ventas_market2['fecha_venta'] = pd.to_datetime(df_ventas_market2['fecha_venta'].dt.date)


In [ ]:
# Agregamos una nueva columna  valor_venta = monto
df_ventas_market2["valor_venta"] = df_ventas_market2["precio_unit"] * df_ventas_market2["cantidad"]

# Agregamos una nueva columna  gasto_venta (si no hay campaña. el costo es cero )
df_ventas_market2["gasto_venta"] = df_ventas_market2["costo_mrkt"] * df_ventas_market2["cantidad"]

# Generamos primero la columna mes
df_ventas_market2["mes"] = df_ventas_market2["fecha_venta"].dt.month

In [ ]:
# Agregar una nueva columna para diferencia: valor_venta - gasto_venta
df_ventas_market2['margen_neto'] = df_ventas_market2['valor_venta'] - df_ventas_market2['gasto_venta']


In [ ]:
df_ventas_market2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2998 entries, 1476 to 744
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_venta      2998 non-null   int64         
 1   id_prod       2998 non-null   int64         
 2   id_producto   2998 non-null   int64         
 3   producto      2998 non-null   object        
 4   rango         2998 non-null   object        
 5   precio_unit   2998 non-null   float64       
 6   cantidad      2998 non-null   int64         
 7   fecha_venta   2998 non-null   datetime64[ns]
 8   id_cat        2998 non-null   int64         
 9   categoria     2998 non-null   object        
 10  precio_rango  2998 non-null   object        
 11  dentro_camp   2998 non-null   int64         
 12  id_camp       2998 non-null   int64         
 13  canal         2998 non-null   object        
 14  costo_mrkt    2998 non-null   float64       
 15  id_canal      2998 non-null   int64      

In [ ]:
df_ventas_market2.head()

,id_venta,id_prod,id_producto,producto,rango,precio_unit,cantidad,fecha_venta,id_cat,categoria,precio_rango,dentro_camp,id_camp,canal,costo_mrkt,id_canal,valor_venta,gasto_venta,mes,margen_neto
1476,410,100,103,Adorno de pared,R3,109.64,3,2024-06-21,1,Decoración,91-120,0,0,no,0.00,0,328.92,0.00,6,328.92
2426,620,100,103,Adorno de pared,R3,92.16,4,2024-10-21,1,Decoración,91-120,0,0,no,0.00,0,368.64,0.00,10,368.64
697,780,100,102,Adorno de pared,R2,79.13,7,2024-03-21,1,Decoración,66-91,1,74,TV,4.81,2,553.91,33.67,3,520.24
255,50,100,102,Adorno de pared,R2,83.10,5,2024-01-31,1,Decoración,66-91,0,0,no,0.00,0,415.50,0.00,1,415.50
112,260,100,103,Adorno de pared,R3,101.48,9,2024-01-15,1,Decoración,91-120,0,0,no,0.00,0,913.32,0.00,1,913.32


# Plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Ale
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Filtrar para el producto específico 'Adorno de pared'
ventas_lampara = df_ventas_market2[df_ventas_market2['producto'] == 'Lámpara de mesa']


In [ ]:
item = 'Lámpara de mesa'
ventas_lampara.head(3)

,producto,fecha_venta,id_venta,canal,cantidad,margen_neto,mes
1,Lámpara de mesa,2024-01-02,811,no,5,525.50,1
62,Lámpara de mesa,2024-01-09,871,no,11,876.04,1
94,Lámpara de mesa,2024-01-13,501,no,6,211.20,1
96,Lámpara de mesa,2024-01-13,669,no,5,483.95,1
110,Lámpara de mesa,2024-01-15,169,no,9,686.88,1


In [ ]:
ventas_lampara = ventas_lampara[["producto", "fecha_venta","id_venta","canal","cantidad", "margen_neto","mes"]]
ventas_lampara.sort_values(by='fecha_venta', ascending=True, inplace=True)


In [ ]:
ventas_mes = ventas_lampara.groupby(['mes', 'canal'], as_index=False)['cantidad'].sum()
ventas_mes  = ventas_mes.sort_values(by='mes', ascending=True)
display(ventas_mes)

,mes,canal,cantidad
0,1,no,120
1,2,no,108
2,3,TV,28
3,3,no,73
4,4,Email,95
5,4,TV,59
6,5,Email,43
7,5,no,70
8,6,no,44
9,7,no,94


In [ ]:
# @title
'''
## -----------------------------------------
# 1. Preparación de datos para evolución de ventas por producto por cantidad
# -----------------------------------------


fig_ventas_cant = px.line(
    ventas_lampara,
    x="fecha_venta",
    y="cantidad",
    color="canal",
    title="Evolución de Unidades por Venta y Canal para " + item
)

fig_ventas_cant.update_layout(width=1200,height=500, xaxis_title="Fecha de Venta", yaxis_title="Unidades")

fig_ventas_cant.show()
'''

In [ ]:
# @title
'''
# -----------------------------------------
# 1. Preparación de datos para evolución de ventas por producto por cantidad
# -----------------------------------------
# POR MES  !!! NO LO QUIERO TOCAR

fig_ventas_cant_mes = px.scatter(
    ventas_mes,
    x="mes",
    y="cantidad",
    color="canal",
    title="Evolución Mensual de Unidades Vendidas por Canal para " + item ,
    labels={
        "mes": "Mes",
        "cantidad": "Unidades Vendidas",
        "canal": "Canal de Venta"
    },
    hover_data={
        "mes": True, # "%d-%m-%Y"
        "cantidad": True,
        "canal": True
    }
)

fig_ventas_cant_mes.update_traces(mode='lines+markers', marker=dict(size=8, opacity=0.7))
fig_ventas_cant_mes.update_layout(
    width=900,
    height=500,
    xaxis_title="mes",
    yaxis_title="Unidades Vendidas",
    title_x=0.5,
    hovermode="x unified"
)
# Forzar los xticks de 1 a 12
fig_ventas_cant_mes.update_xaxes(dtick=1, tick0=1, range=[0.5, 12.5])

fig_ventas_cant_mes.show()
'''

POR FECHA

In [ ]:
# POR FECHA
# Muestra el gráfico fig_ventas_cant, que ya está definido para `ventas_lampara` sin agrupar por mes.
fig_ventas_cant = px.scatter(
    ventas_lampara,
    x="fecha_venta",
    y="cantidad",
    color="canal",
    title="**Evolución de Unidades por Venta y Canal para " + item + "**",
    labels={
        "fecha_venta": "Fecha de Venta",
        "cantidad": "Unidades Vendidas",
        "canal": "Canal de Venta"
    },
    hover_data={
        "fecha_venta": "%Y-%m-%d",
        "cantidad": True,
        "canal": True
    }
)

fig_ventas_cant.update_traces(mode='lines+markers', marker=dict(size=8, opacity=0.7))
fig_ventas_cant.update_layout(
    width=1200,
    height=600,
    xaxis_title="Fecha de Venta",
    yaxis_title="Unidades Vendidas",
    title_x=0.5,
    hovermode="x unified"
)

fig_ventas_cant.show()


### OLD CODE

In [ ]:
# @title
'''
# -----------------------------------------
# 1. Preparación de datos para evolución de ventas por producto
# -----------------------------------------

# La variable 'ventas_adorno' ya está definida en una celda anterior como:
# ventas_adorno = df_ventas_market[df_ventas_market['producto'] == 'Adorno de pared']
# Contiene todas las ventas para 'Adorno de pared', tanto con como sin campañas.

# ---------------------------------------------
# 2. Gráfico de evolución del margen neto por cada venta
# ---------------------------------------------
fig_margen_neto_por_venta = px.line(
    ventas_adorno,
    x="fecha_venta",
    y="margen_neto",
    color="canal",
    title="Evolución del Margen Neto por Venta y Canal para Adorno de pared"
)

fig_margen_neto_por_venta.update_layout(width=1000, xaxis_title="Fecha de Venta", yaxis_title="Margen Neto")

fig_margen_neto_por_venta.show()
'''

In [ ]:
# @title
'''
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Definimos una función que generará el gráfico
def generar_grafico_interactivo( canal_select, tipo_grafico):
    df_filtrado = ventas_adorno[ventas_adorno['canal'] == canal_select]

    if tipo_grafico == 'dispersion':
       fig = px.scatter(
             df_filtrado,
             x="fecha_venta",
             y="margen_neto",
             color="canal",
             title="Dispersión del Margen Neto por Venta y Canal para Adorno de pared",
             hover_data=['id_venta', 'margen_neto', 'cantidad'] # Add more details on hover
       )
    elif tipo_grafico == 'linea':
        fig = px.line(
            df_filtrado,
            x="fecha_venta",
            y="margen_neto",
            color="canal",
            title="Evolución del Margen Neto por Venta y Canal para Adorno de pared",
            hover_data=['id_venta', 'margen_neto', 'cantidad']
        )
    else:
        print("Tipo de gráfico no reconocido.")
        return

    fig.update_layout(width=900, xaxis_title="Fecha de Venta", yaxis_title="Margen Neto")
    fig.show()
    #return fig # Return the figure object instead of showing it

#generar_grafico_interactivo( "no", "dispersion")
# Usamos interact para vincular los widgets a nuestra función
# Cada vez que cambies una opción en los widgets, la función se re-ejecutará

# Creamos los widgets para las variables que queremos cambiar

selector_canal = widgets.Dropdown(
    options=ventas_adorno['canal'].unique().tolist(),
    value='no',
    description='Canal:'
)

selector_tipo_grafico = widgets.RadioButtons(
    options=['linea', 'dispersion'],
    value='dispersion',
    description='Tipo de Gráfico:'
)

# The interactive widget itself is the last expression, so it will be displayed.
#widgets.interactive( generar_grafico_interactivo, canal_select=selector_canal, tipo_grafico=selector_tipo_grafico)

interactive_widget = widgets.interactive(
    generar_grafico_interactivo,
    canal_select=selector_canal,
    tipo_grafico=selector_tipo_grafico)

display(interactive_widget)
'''


.

# &emsp; &emsp; &emsp; GRAFICOS INTERACTIVOS CON PLOTLY

.


In [ ]:
import pandas as pd

df_marketing.head(3)

,id_camp,producto,canal,costo_mrkt,id_canal,fecha_ini,fecha_fin,id_prod,id_cat,media_mrkt
0,14,Adorno de pared,RRSS,4.16,1,2024-10-22,2024-12-21,100,1,4.683333
1,74,Adorno de pared,TV,4.81,2,2024-03-20,2024-05-03,100,1,4.683333
2,44,Adorno de pared,Email,5.08,3,2024-04-13,2024-05-10,100,1,4.683333


In [ ]:
df_markt_canal = df_marketing[["producto", "canal", "fecha_ini", "fecha_fin"]]
df_markt_canal.head()

<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 89
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   producto   90 non-null     object        
 1   canal      90 non-null     object        
 2   fecha_ini  90 non-null     datetime64[ns]
 3   fecha_fin  90 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.5+ KB


<br>

### PRODUCTOS TOP

In [ ]:
data = {
  "producto": ["Lámpara de mesa","Auriculares", "Microondas","Cafetera", "Cuadro decorativo"],
  "cant_vend": [1112, 958, 912, 765, 726],
  "valor_venta":[82276.38, 74175.58, 72562.89, 59607.31, 54297.60]
}
productos_top = pd.DataFrame(data)
productos_top.info()
productos_top.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   producto     5 non-null      object 
 1   cant_vend    5 non-null      int64  
 2   valor_venta  5 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 252.0+ bytes


,producto,cant_vend,valor_venta
0,Lámpara de mesa,1112,82276.38
1,Auriculares,958,74175.58
2,Microondas,912,72562.89
3,Cafetera,765,59607.31
4,Cuadro decorativo,726,54297.60


In [ ]:
# Graficamos el boxplotpor cantidad vendida

fig = px.box(productos_top,
       y='cant_vend',
       title='Boxplot por Cantidad vendida',
       hover_data=['cant_vend'],
       )
fig.update_traces(fillcolor='#ffffcc', line=dict(color='navy', width=2))

fig.show()



In [ ]:
# Obtenemos una lista de todos los productos (unique)
lista_prod = df_aLista_prod["Producto"].tolist()
print(lista_prod)

lista_top = productos_top["producto"].tolist()
print(lista_top)

['Adorno de pared', 'Alfombra', 'Aspiradora', 'Auriculares', 'Batidora', 'Cafetera', 'Candelabro', 'Consola de videojuegos', 'Cortinas', 'Cuadro decorativo', 'Cámara digital', 'Elementos de cerámica', 'Espejo decorativo', 'Freidora eléctrica', 'Heladera', 'Horno eléctrico', 'Jarrón decorativo', 'Laptop', 'Lavadora', 'Lámpara de mesa', 'Microondas', 'Parlantes Bluetooth', 'Plancha de vapor', 'Proyector', 'Rincón de plantas', 'Secadora', 'SmartWatch', 'Smartphone', 'Tablet', 'Televisor']
['Lámpara de mesa', 'Auriculares', 'Microondas', 'Cafetera', 'Cuadro decorativo']


In [ ]:

def get_chdates( prod, chan):

    # busca producto y canal y devuelve una tupla con (fecha_ini, fecha_fin)
    # index[0] => primer registro encontrado
    idx = df_markt_canal.query("producto == @prod & canal == @chan").index[0]

    # df_markt_canal.iloc[ idx, 2], df_markt_canal.iloc[ idx, 3]
    fech_ini = df_markt_canal.loc[ idx, "fecha_ini"].date()
    fech_fin = df_markt_canal.loc[ idx, "fecha_fin"].date()
    return( fech_ini, fech_fin)

In [ ]:
# Obtenemos una lista de todos los canales (unique)
available_channels = ['All'] + sorted(ventas_lampara['canal'].unique().tolist())
print(available_channels)

['All', 'Email', 'RRSS', 'TV', 'no']


## Filtrar Datos de Ventas por Rango de Fecha (rango de campaña de marketing)  

Filtramos el DataFrame `ventas_lampara` para incluir solamente los regiastros de ventas donde `fecha_venta` esta entre '2024-04-01' y '2024-07-01' (inclusive). <br>
Esto creará un nuevo DataFrame conteniendo solamente los datos para un periodo de 3 meses (aproxim. el lapso de la campaña de marketing).  


In [ ]:

prod = "Lámpara de mesa" # item
chan = "RRSS"
# obtiene tupla con fecha ini y fin del canal mrkt
start_date, end_date = get_chdates( prod, chan)
end_date = pd.to_datetime(end_date) + pd.DateOffset(days=1) # sumo un mes al final
end_date = end_date.date()

print(start_date, end_date)

2024-05-30 2024-06-30


# Período de 3 meses

Se visualiza mejor si segmentamos el ploteo al lapso de una campaña que si lo hacemos anual

In [ ]:

# start_date = pd.to_datetime('2024-04-01')
# end_date = pd.to_datetime('2024-07-01')

start_date = pd.to_datetime('2024-10-01')
end_date = pd.to_datetime('2025-01-01')

ventas_lampara_filtered = ventas_lampara[
    (ventas_lampara['fecha_venta'] >= start_date) &
    (ventas_lampara['fecha_venta'] <= end_date)
]

print(f"Original ventas_lampara shape: {ventas_lampara.shape}")
print(f"Filtered ventas_lampara_filtered shape: {ventas_lampara_filtered.shape}")

display(ventas_lampara_filtered.head())

Original ventas_lampara shape: (176, 7)
Filtered ventas_lampara_filtered shape: (38, 7)


,producto,fecha_venta,id_venta,canal,cantidad,margen_neto,mes
2290,Lámpara de mesa,2024-10-01,521,no,6,290.64,10
2306,Lámpara de mesa,2024-10-03,681,no,1,113.04,10
2342,Lámpara de mesa,2024-10-08,991,no,12,955.56,10
2425,Lámpara de mesa,2024-10-21,591,no,1,74.62,10
2550,Lámpara de mesa,2024-11-06,719,RRSS,2,104.54,11


In [ ]:

fig_ventas_cant = px.scatter(
    ventas_lampara_filtered,
    x="fecha_venta",
    y="cantidad",
    color="canal",
    title="Evolución de Unidades por Venta y Canal para "+ item+ " (2024-04-01 to 2024-07-01)",
    labels={
        "fecha_venta": "Fecha de Venta",
        "cantidad": "Unidades Vendidas",
        "canal": "Canal de Venta"
    },
    hover_data={
        "fecha_venta": True, #"%Y-%m-%d",
        "cantidad": True,
        "canal": True
    }
)

fig_ventas_cant.update_traces(mode='lines+markers', marker=dict(size=8, opacity=0.7))
fig_ventas_cant.update_layout(
    width=1200,
    height=500,
    xaxis_title="Fecha de Venta",
    yaxis_title="Unidades Vendidas",
    title_x=0.5,
    hovermode="x unified"
)

fig_ventas_cant.show()

# 3 MESES !!!


###
*   El DataFrame `ventas_lampara` fue filtrado para incluir registros de ventas entre '2024-04-01' y '2024-07-01'. <br>
*   El DataFrame original contiene 176 filas, mientras que el DataFrame filtrado, `ventas_lampara_filtered`, consiste de 52 registros de ventas para el periodo epecifiedo de tres meses. <br>
*   Un grafico de dispersion  `px.scatter` fué generated, effectively visualizing la evolucion de las ventas de 'Lámpara de mesa' para  'fecha\_venta' and 'cantidad', segmentedo por 'canal' (canal de ventas) durante el periodo defined.




.

.


In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

<BR>


In [ ]:
# @title
'''
# Inspeccionar df_marketing para 'Lámpara de mesa' y canal 'RRSS'
df_marketing_lampara_rrss = df_marketing[
    (df_marketing['producto'] == item) &
    (df_marketing['canal'] == 'RRSS')
]

print(f"Campañas de marketing para '{item}' en el canal 'RRSS':")
display(df_marketing_lampara_rrss)

# Inspeccionar el rango de fechas de ventas en ventas_lampara para 'RRSS'
ventas_lampara_rrss = ventas_lampara[ventas_lampara['canal'] == 'RRSS']
print(f"\nFechas de venta en ventas_lampara para '{item}' en el canal 'RRSS':")
print(f"Fecha mínima de venta: {ventas_lampara_rrss['fecha_venta'].min()}")
print(f"Fecha máxima de venta: {ventas_lampara_rrss['fecha_venta'].max()}")
'''

Campañas de marketing para 'Lámpara de mesa' en el canal 'RRSS':


,id_camp,producto,canal,costo_mrkt,id_canal,fecha_ini,fecha_fin,id_prod,id_cat,media_mrkt
59,2,Lámpara de mesa,RRSS,5.88,3,2024-05-30,2024-06-29,290,1,5.31



Fechas de venta en ventas_lampara para 'Lámpara de mesa' en el canal 'RRSS':
Fecha mínima de venta: 2024-11-06 00:00:00
Fecha máxima de venta: 2024-12-18 00:00:00


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
# from google.colab import output
# output.disable_custom_widget_manager()

<br>

<br>

## Estatico para Ventas de Lampara de Mesa

 Paso 1: Inicializar FigureWidget y Preparar Datos Base

In [ ]:

ventas_lampara['fecha_venta'] = pd.to_datetime(ventas_lampara['fecha_venta'])
df_marketing['fecha_ini'] = pd.to_datetime(df_marketing['fecha_ini'])
df_marketing['fecha_fin'] = pd.to_datetime(df_marketing['fecha_fin'])

# 2. Determine the overall minimum and maximum 'fecha_venta' from ventas_lampara
min_date = ventas_lampara['fecha_venta'].min()
max_date = ventas_lampara['fecha_venta'].max()

# 3. Initialize an empty go.FigureWidget and assign it to 'f'
f = go.FigureWidget()

# 4. Set the layout properties of the FigureWidget
f.layout.title = "Evolución de Unidades por Venta y Canal para " + item
f.layout.xaxis.title = "Fecha de Venta"
f.layout.yaxis.title = "Unidades Vendidas"
f.layout.hovermode = "x unified"
f.layout.width = 1200
f.layout.height = 600

print("FigureWidget initializado")


FigureWidget initializado


In [ ]:
#

def update_plot3(selected_channel):
    # Always display the full date range of ventas_lampara
    start_date_plot = ventas_lampara['fecha_venta'].min()
    end_date_plot = ventas_lampara['fecha_venta'].max()

    # Filter ventas_lampara by the full date range
    filtered_by_dates_df = ventas_lampara[
        (ventas_lampara['fecha_venta'] >= start_date_plot) &
        (ventas_lampara['fecha_venta'] <= end_date_plot)
    ].copy()

    # Simplify channel filtering logic
    if selected_channel == 'All':
        final_filtered_df = filtered_by_dates_df  # Show all channels
    else:
        # Show only the selected channel's sales data
        final_filtered_df = filtered_by_dates_df[
            filtered_by_dates_df['canal'] == selected_channel
        ]

    # Agrupar por fecha_venta y canal para agregar cantidades
    grouped_df = final_filtered_df.groupby(['fecha_venta', 'canal'], as_index=False)['cantidad'].sum()

    # Actualizar traces del gráfico
    with f.batch_update():
        f.data = []  # Limpiar traces existentes

        if not grouped_df.empty:
            for c in grouped_df['canal'].unique():
                channel_data = grouped_df[grouped_df['canal'] == c]
                f.add_trace(go.Scatter(
                    x=channel_data['fecha_venta'],
                    y=channel_data['cantidad'],
                    mode='lines+markers',
                    name=c,
                    marker=dict(size=8, opacity=0.7),
                    hovertemplate=
                        '<b>Fecha</b>: %{x|%Y-%m-%d}<br>' +
                        '<b>Unidades</b>: %{y}<br>' +
                        '<b>Canal</b>: %{customdata[0]}<extra></extra>',
                    customdata=channel_data[['canal']]
                ))
        else:
            # Puedes agregar un mensaje o anotación si no hay datos
            pass

print("Update function 'update_plot' modified for simplified filtering and full date range display.")


Update function 'update_plot' modified for simplified filtering and full date range display.


<BR>

In [ ]:
channel_widget = widgets.Dropdown(
    options=available_channels,
    value='All',
    description='Canal:',
    disabled=False,
)

print("** ULTIMO **  selecciona 'channel_widget' ")

** ULTIMO **  selecciona 'channel_widget' 


In [ ]:
# @title

interactive_plot_output = widgets.interactive_output(
    update_plot3,
    {
        'selected_channel': channel_widget
    }
)

print(f"Interactive plot for '{item}' sales data with dynamic channel-based filtering.")

# Display widgets and the plot
display(widgets.VBox([widgets.HBox([channel_widget]), f]))


Interactive plot for 'Lámpara de mesa' sales data with dynamic channel-based filtering.


<br>


<br>

<br>

## Paso 1: Inicializar FigureWidget y Preparar Datos Base



In [ ]:
# @title
'''
ventas_lampara['fecha_venta'] = pd.to_datetime(ventas_lampara['fecha_venta'])
df_marketing['fecha_ini'] = pd.to_datetime(df_marketing['fecha_ini'])
df_marketing['fecha_fin'] = pd.to_datetime(df_marketing['fecha_fin'])

# 2. Determine the overall minimum and maximum 'fecha_venta' from ventas_lampara
min_date = ventas_lampara['fecha_venta'].min()
max_date = ventas_lampara['fecha_venta'].max()

# 3. Initialize an empty go.FigureWidget and assign it to 'f'
f = go.FigureWidget()

# 4. Set the layout properties of the FigureWidget
f.layout.title = "Evolución de Unidades por Venta y Canal para " + item
f.layout.xaxis.title = "Fecha de Venta"
f.layout.yaxis.title = "Unidades Vendidas"
f.layout.hovermode = "x unified"
f.layout.width = 1200
f.layout.height = 600

print("FigureWidget initializado")
'''

FigureWidget initializado


## Paso 2: Definir la Función 'update_plot'


In [ ]:
def update_plot(selected_channel):

    # Always display the full date range of ventas_lampara
    start_date_plot = ventas_lampara['fecha_venta'].min()
    end_date_plot = ventas_lampara['fecha_venta'].max()

    # Filter ventas_lampara by the full date range
    filtered_by_dates_df = ventas_lampara[
        (ventas_lampara['fecha_venta'] >= start_date_plot) &
        (ventas_lampara['fecha_venta'] <= end_date_plot)
    ].copy()

    # Simplify channel filtering logic
    if selected_channel == 'All':
        final_filtered_df = filtered_by_dates_df  # Show all channels
    else:
        # Show only the selected channel's sales data
        final_filtered_df = filtered_by_dates_df[
            filtered_by_dates_df['canal'] == selected_channel
        ]

    # Agrupar por fecha_venta y canal para agregar cantidades
    grouped_df = final_filtered_df.groupby(['fecha_venta', 'canal'], as_index=False)['cantidad'].sum()

    # Actualizar traces del gráfico
    with f.batch_update():
        f.data = []  # Limpiar traces existentes

        if not grouped_df.empty:
            for c in grouped_df['canal'].unique():
                channel_data = grouped_df[grouped_df['canal'] == c]
                f.add_trace(go.Scatter(
                    x=channel_data['fecha_venta'],
                    y=channel_data['cantidad'],
                    mode='lines+markers',
                    name=c,
                    marker=dict(size=8, opacity=0.7),
                    hovertemplate=
                        '<b>Fecha</b>: %{x|%Y-%m-%d}<br>' +
                        '<b>Unidades</b>: %{y}<br>' +
                        '<b>Canal</b>: %{customdata[0]}<extra></extra>',
                    customdata=channel_data[['canal']]
                ))
        else:
            # Puedes agregar un mensaje o anotación si no hay datos
            pass

print("** ULTIMO **  'update_plot")


In [ ]:
print(lista_prod[0])

prod_widget = widgets.Dropdown(
    options=lista_top,
    value=lista_top[0],
    description='Producto:',
    disabled=False,
)

print("** ULTIMO **  seleccion 'prod_widget' .")

Adorno de pared
** ULTIMO **  seleccion 'prod_widget' .


## Paso 3: Crear el Widget Selector de Canal (Completado)


## Paso 4: Enlazar Widgets y Mostrar el Gráfico Interactivo

Conectar el widget desplegable para selección del canal a la funcion de actualizacion de graficado (plot_update)  y mostrar el grafico interactivo.


In [ ]:
interactive_plot_output = widgets.interactive_output(
    update_plot,
    {
        'selected_channel': channel_widget
    }
)
print(f"** ULTIMO ** Plot Interactivo para '{item}' con canal dinamico")

# Display widgets y plot
display(widgets.VBox([widgets.HBox([channel_widget]), f]))

### El próximo paso sería incorporar al selector el nombre del producto para los 5 productos top y así poder visualizarlos por cada canal y por tanto para distintas fechas.   <br>
### Hasta ahora utilizamos las cantidades vendidas para comparar las ventas, y también se podria hacer por margen neto de ventas.

In [ ]:

def get_ventas_xprod( item ):
    # devuelve el df para un item determinado y ordenado por fecha venta
    global vent_xprod
    vent_xprod = df_ventas_market2[df_ventas_market2['producto'] == item]
    vent_xprod = vent_xprod[["producto", "fecha_venta","id_venta","canal","cantidad", "margen_neto","mes"]]
    vent_xprod = vent_xprod.sort_values(by='fecha_venta', ascending=True)
    return vent_xprod


In [ ]:
# Funcion update combinada de producto y canal

def update_plot2(selected_prod, selected_channel):

    # Filtrar para el producto específico selected_prod
    ventasxprod = get_ventas_xprod( selected_prod)

    # Siempre mostrar el rango completo de fechas para el prod elegido
    start_date_plot = ventasxprod['fecha_venta'].min()
    end_date_plot = ventasxprod['fecha_venta'].max()

    # Filter ventas_lampara by the full date range
    filtered_by_dates_df = ventasxprod[
        (ventasxprod['fecha_venta'] >= start_date_plot) &
        (ventasxprod['fecha_venta'] <= end_date_plot)
    ].copy()

    # Simplificar logica de filtrado de canal
    if selected_channel == 'All':
        final_filtered_df = filtered_by_dates_df  # Show all channels
    else:
        # Show only the selected channel's sales data
        final_filtered_df = filtered_by_dates_df[
            filtered_by_dates_df['canal'] == selected_channel
        ]

    # Agrupar por fecha_venta y canal para agregar cantidades
    grouped_df = final_filtered_df.groupby(['fecha_venta', 'canal'], as_index=False)['cantidad'].sum()

    # Actualizar traces del gráfico
    with f.batch_update():
        f.data = []  # Limpiar traces existentes

        if not grouped_df.empty:
            for c in grouped_df['canal'].unique():
                channel_data = grouped_df[grouped_df['canal'] == c]
                f.add_trace(go.Scatter(
                    x=channel_data['fecha_venta'],
                    y=channel_data['cantidad'],
                    mode='lines+markers',
                    name=c,
                    marker=dict(size=8, opacity=0.7),
                    hovertemplate=
                        '<b>Fecha</b>: %{x|%Y-%m-%d}<br>' +
                        '<b>Unidades</b>: %{y}<br>' +
                        '<b>Canal</b>: %{customdata[0]}<extra></extra>',
                    customdata=channel_data[['canal']]
                ))
        else:
            # Puedes agregar un mensaje o anotación si no hay datos
            pass

print("** ULTIMO **  'update_plot2")


** ULTIMO **  'update_plot2


In [ ]:


interactive_plot_output = widgets.interactive_output(
    update_plot2, {'selected_prod': prod_widget, 'selected_channel': channel_widget}
)
print(f" Plot Interactivo para '{item}' con canal dinamico")

# Display widgets y plot
display(widgets.VBox([widgets.HBox([prod_widget, channel_widget]), f]))

 Plot Interactivo para 'Lámpara de mesa' con canal dinamico


FALTA AJUSTARLO PARA QUE MUESTRE 3 MESES QUE ES LO QUE DURA CADA CANAL

Se puede observar visualmente que las ventas fuera de la campaña de marketing (aprox. 2/3), <br>
son mucho mayores que dentro de la campaña (1/3). <br>
Queda más patente cuando uno ve que cada campaña de marketing por canal, dura alrededor de 2 meses.